# THERMAL fixtures from real TORAX

Generates the facility profiles for `apps/thermal-investigation` by running
**TORAX 1.4.3** on the repo's real device registry, instead of the reduced 1-D
transport solve the app falls back to.

Run this on Colab because TORAX cannot be installed on an Intel Mac:
`torax==1.4.3` needs `jax>=0.10.0`, and jaxlib's last macOS **x86_64** wheel is
0.4.38. Colab is Linux x86_64, where the wheels exist.

Runtime > Change runtime type > CPU is fine. GPU does not help: TORAX defaults
to float64, which is slow on consumer NVIDIA cards for this 1-D problem.

In [ ]:
#@title 1. Install TORAX (a few minutes; JAX is large)
!pip install -q "torax==1.4.3" "numpy>=1.26" "scipy>=1.13"
import torax, jax
print("torax", torax.__version__, "| jax", jax.__version__, "| devices", jax.devices())

In [ ]:
#@title 2. Clone the branch
BRANCH = "codex/thermal-investigation"  #@param {type:"string"}
REPO = "https://github.com/wolfeiq/tokamak-flower-berlin-2026.git"  #@param {type:"string"}
!git clone --branch $BRANCH --depth 1 $REPO repo
%cd repo
!ls scripts/

In [ ]:
#@title 3. Sanity-check the device registry before spending compute
import sys; sys.path.insert(0, ".")
from hfmarl.devices.registry import DEVICES
from hfmarl.envs.torax_config import pedestal_T_keV
for name in ("diiid_like", "sparc_like", "tcv_like"):
    d = DEVICES[name]
    print(f"{name:<12} R={d.R_major:<5} a={d.a_minor:<5} B0={d.B_0:<6} "
          f"Ip={d.Ip_nominal:.2e}  T_ped={pedestal_T_keV(d):.2f} keV")

## Generate

`constant` transport is fast but makes every device look alike in chi, which
defeats the point of comparing facilities. `qlknn` is the physically meaningful
choice and is markedly slower to compile the first time.

If a step fails, the error names what TORAX actually exposed — paste it back
rather than guessing; the source profile is the one thing this demo cannot
afford to get silently wrong, since the whole investigation is about commanded
versus delivered power.

In [ ]:
#@title 4. Run TORAX for all three facilities
TRANSPORT = "qlknn"  #@param ["qlknn", "constant"]
!python scripts/generate_thermal_fixtures.py --transport $TRANSPORT

In [ ]:
#@title 5. Inspect what came out
import json, numpy as np
fx = json.load(open("apps/thermal-investigation/thermal_investigation/_fixtures.json"))
for site, f in sorted(fx.items()):
    T = np.asarray(f["T_e_keV"]); S = np.asarray(f["source"])
    print(f"{site} {f['device']:<12} {f['provenance']:<22} "
          f"T_axis={T[0]:7.3f} keV  T_edge={T[-1]:6.3f}  P_int={np.trapezoid(S):.3e}")

In [ ]:
#@title 6. Plot the profiles
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for site, f in sorted(fx.items()):
    ax[0].plot(f["rho"], f["T_e_keV"], label=f"{site} ({f['device']})")
    ax[1].plot(f["rho"], f["source"], label=site)
ax[0].set_xlabel("rho_norm"); ax[0].set_ylabel("T_e [keV]"); ax[0].set_title("Temperature")
ax[1].set_xlabel("rho_norm"); ax[1].set_ylabel("S [W/m^3]"); ax[1].set_title("Electron heating")
ax[0].legend(); ax[1].legend(); plt.tight_layout(); plt.show()

In [ ]:
#@title 7. Download the fixtures, then commit them to the branch
from google.colab import files
files.download("apps/thermal-investigation/thermal_investigation/_fixtures.json")

## After downloading

Drop `_fixtures.json` into
`apps/thermal-investigation/thermal_investigation/` and commit it. The app
prefers it automatically and reports `provenance: torax-1.4.3-qlknn` in every
released finding, so a report built on TORAX is distinguishable from one built
on the reduced solve.

Re-run this whenever the device registry or the TORAX pin changes.